In [11]:
import requests
import time
import pandas as pd
from requests.exceptions import ConnectionError, Timeout, ChunkedEncodingError

ESEARCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
MIN_DELAY = 0.35  # NCBI recommends max 3 requests/sec, use 0.35s for safety

def make_request_with_retry(session, url, params, max_retries=7, last_request_time=[0]):
    """Make a request with exponential backoff and intelligent rate limiting."""
    # Enforce minimum delay between requests
    elapsed = time.time() - last_request_time[0]
    if elapsed < MIN_DELAY:
        time.sleep(MIN_DELAY - elapsed)
    
    for attempt in range(max_retries):
        try:
            r = session.get(url, params=params, timeout=60)
            last_request_time[0] = time.time()
            
            if r.status_code == 429:
                wait_time = (2 ** attempt) * 2
                print(f"    Rate limited (429). Waiting {wait_time}s...")
                time.sleep(wait_time)
                continue
            elif r.status_code == 500:
                # Handle NCBI server errors
                wait_time = (2 ** attempt) * 4
                if attempt < max_retries - 1:
                    print(f"    Server error (500). Waiting {wait_time}s before retry {attempt + 1}/{max_retries}...")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"    Server error (500) after {max_retries} retries. Skipping this query.")
                    return None
            r.raise_for_status()
            return r
        except (ConnectionError, Timeout, ChunkedEncodingError) as e:
            wait_time = (2 ** attempt) * 3
            if attempt < max_retries - 1:
                print(f"    Connection error ({type(e).__name__}). Retrying in {wait_time}s...")
                time.sleep(wait_time)
                continue
            else:
                print(f"    Connection error after {max_retries} retries. Skipping this query.")
                return None
        except requests.exceptions.HTTPError as e:
            if "429" in str(e):
                wait_time = (2 ** attempt) * 2
                time.sleep(wait_time)
                continue
            else:
                print(f"    HTTP Error: {e}. Skipping this query.")
                return None
    print(f"Max retries exceeded. Skipping this query.")
    return None

def get_pmids_with_count(session, query, retmax=9999):
    """Fetch PMIDs directly (one API call instead of two)."""
    params = {
        "db": "pubmed",
        "term": query,
        "retmode": "json",
        "retmax": retmax
    }
    r = make_request_with_retry(session, ESEARCH_URL, params)
    if r is None:
        return 0, []  # Return empty result if request failed
    try:
        data = r.json()["esearchresult"]
        return int(data.get("count", 0)), data.get("idlist", [])
    except Exception as e:
        print(f"    Error parsing response: {e}. Returning empty result.")
        return 0, []

def fetch_pmids_recursive(session, base_query, year, month=None, day=None, all_pmids=None):
    """Recursively fetch PMIDs, breaking down by month/day if count > 9999."""
    if all_pmids is None:
        all_pmids = set()
    
    # Build date filter
    if day is not None:
        date_filter = f'("{year}/{month:02d}/{day:02d}"[PDAT])'
    elif month is not None:
        date_filter = f'("{year}/{month:02d}/01"[PDAT] : "{year}/{month:02d}/31"[PDAT])'
    else:
        date_filter = f'("{year}"[PDAT])'
    
    query = f'{base_query} AND {date_filter}'
    count, pmids = get_pmids_with_count(session, query, retmax=9999)
    
    if count == 0:
        return all_pmids
    
    all_pmids.update(pmids)
    
    if count <= 9999:
        level = f"{year}" + (f"-{month:02d}" if month else "") + (f"-{day:02d}" if day else "")
        print(f"  {level}: {len(pmids)} PMIDs (total: {len(all_pmids)})")
        return all_pmids
    
    # Need to break down further
    if month is None:
        print(f"  Year {year} has {count} results, splitting by month...")
        for m in range(1, 13):
            fetch_pmids_recursive(session, base_query, year, month=m, all_pmids=all_pmids)
    elif day is None:
        print(f"  {year}-{month:02d} has {count} results, splitting by day...")
        days_in_month = 31
        if month in [4, 6, 9, 11]:
            days_in_month = 30
        elif month == 2:
            if (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0):
                days_in_month = 29
            else:
                days_in_month = 28
        
        for d in range(1, days_in_month + 1):
            fetch_pmids_recursive(session, base_query, year, month=month, day=d, all_pmids=all_pmids)
    else:
        print(f"  WARNING: {year}-{month:02d}-{day:02d} has {count} (>9999), truncated")
    
    return all_pmids

# --- Main execution ---
year_range = (1960, 2025)  # 20 years: captures modern AMR literature (1.5-2 hours with optimizations)

search_queries = [
    # General resistance with gene/mutation
    '("drug resistance, microbial"[MeSH Terms] OR "antimicrobial resistance"[All Fields]) AND (gene[All Fields] OR mutation[All Fields])',
    '("antibiotic resistance"[Title/Abstract] OR "multidrug resistance"[Title/Abstract]) AND (gene OR mutation)',
    '"drug resistance, microbial"[MeSH Terms] AND ("genes"[MeSH Terms] OR "mutation"[MeSH Terms])',
    '"antimicrobial resistance"[Title/Abstract]',
    # Resistance + bacteria
    '(("resistance"[MeSH Terms] OR "resistance"[All Fields]) AND ("bacteria"[MeSH Terms] OR "bacteria"[All Fields] OR "bacterial"[All Fields]))',
    
    # Partial word matching - MeSH terms containing "bacterial" AND "genetics"
    '(bacterial[mh] AND genetics[sh])',
    '(bacteria[mh] AND genetics[sh])',
    '(Plasmids[mh] AND genetics[sh])',
    
    # Beta-lactamases/Cephalosporinase with genetics subheading
    '(beta-Lactamases[mh] AND genetics[sh])',
    '(Cephalosporinase[mh] AND genetics[sh])',
    
    # Bacterial Proteins with genetics
    '("Bacterial Proteins"[mh] AND genetics[sh])',
    
    # Enterobacteriaceae with genetics
    '(Enterobacteriaceae[mh] AND genetics[sh])',
    
    # Salmonella with genetics
    '(Salmonella[mh] AND genetics[sh])',
    
    # Gene Expression Regulation, Bacterial
    '"Gene Expression Regulation, Bacterial"[mh]',
    
    # Broad: any MeSH containing "resistance" AND any containing "genetics"
    '(resistance[mh] AND genetics[mh])',
    
    # Drug Resistance with genetics subheading
    '("Drug Resistance"[mh] AND genetics[sh])',
    
    # Anti-Bacterial Agents with genetics
    '("Anti-Bacterial Agents"[mh] AND genetics[sh])',
    
    # Search for Mutation as a MeSH heading
    'Mutation[mh]',
    
    # Drug Resistance, Microbial combined with bacteria/genetics
    '("Drug Resistance, Microbial"[mh] AND (bacterial[mh] OR bacteria[mh] OR genetics[mh]))',
    
    # Articles with Drug Resistance AND any bacterial term
    '("Drug Resistance, Microbial"[mh] AND (Bacillus[mh] OR Staphylococcus[mh] OR Escherichia[mh]))',
    
    # Bacteria AND genetics in any combination
    '((bacterial[mh] OR bacteria[mh]) AND genetics[mh])',
    
    # Mutation combined with Plasmids
    '(Mutation[mh] AND Plasmids[mh])',
    
    # Anti-Bacterial Agents with Mutation
    '("Anti-Bacterial Agents"[mh] AND Mutation[mh])',
    
    # Drug Resistance with Mutation
    '("Drug Resistance, Microbial"[mh] AND Mutation[mh])',

    # Cephalosporins (broad class)
    '("Cephalosporins"[mh] OR "Cephalosporin Resistance"[mh]) AND ("Drug Resistance, Microbial"[mh] OR "beta-Lactamases"[mh])',
    
    # Specific cephalosporins
    '(Cefpodoxime[mh] OR Ceftizoxime[mh]) AND ("beta-Lactamases"[mh] OR "Drug Resistance, Microbial"[mh])',
    
    # Beta-lactamase inhibitors
    '("beta-Lactamase Inhibitors"[mh]) AND ("beta-Lactamases"[mh] OR "Drug Resistance, Microbial"[mh])',
    
    # Beta-lactamases with biosynthesis focus
    '("beta-Lactamases"[mh] AND biosynthesis[sh]) AND ("bacteria"[mh] OR "Enterobacteriaceae"[mh])',
    
    # Klebsiella-specific resistance
    '("Klebsiella pneumoniae"[mh] OR "Klebsiella Infections"[mh]) AND ("beta-Lactamases"[mh] OR "Drug Resistance, Microbial"[mh])',
    
    # Broad antibiotic classes
    '("Anti-Bacterial Agents"[mh] AND ("Drug Resistance, Microbial"[mh] OR "beta-Lactamases"[mh]))',
    
    # Carbapenems
    '("Carbapenems"[mh] OR "Carbapenem Resistance"[mh]) AND ("beta-Lactamases"[mh] OR "Drug Resistance, Microbial"[mh])',
    
    # Enzymatic resistance mechanisms
    '("beta-Lactamases"[mh] AND (biosynthesis[sh] OR physiology[sh] OR isolation[sh]))',
    
    # Aminoglycosides
    '("Aminoglycosides"[mh] OR "Aminoglycoside Resistance"[mh]) AND ("Drug Resistance, Microbial"[mh])',
    
    # Fluoroquinolones
    '("Fluoroquinolones"[mh] OR "Quinolone Resistance"[mh]) AND ("Drug Resistance, Microbial"[mh])',
    
    # --- GENERIC ABSTRACT SEARCHES FOR MECHANISM-FOCUSED PAPERS WITHOUT FULL MESH INDEXING ---
    
    # Generic target site mutations + antibiotic/resistance (catches gyrA/parC, 23S rRNA, rpoB, fusA, etc.)
    '(("target site mutation" OR "target mutation" OR "point mutation")[Title/Abstract] AND ("antibiotic" OR "drug" OR "resistance")[Title/Abstract])',
    
    # Generic resistance genes + mutation (catches any gene-specific resistance)
    '(("gyrA" OR "parC" OR "gyrB" OR "parE" OR "23S" OR "rpoB" OR "rpoC" OR "fusA" OR "ftsZ" OR "murA" OR "murB" OR "walK" OR "walR" OR "pbp")[Title/Abstract] AND ("mutation" OR "polymorphism" OR "variant")[Title/Abstract])',
    
    # Generic DNA/protein target names + resistance
    '(("DNA gyrase" OR "topoisomerase" OR "RNA polymerase" OR "ribosomal protein" OR "penicillin binding protein" OR "murein biosynthesis")[Title/Abstract] AND ("mutation" OR "resistance")[Title/Abstract])',
    
    # Generic resistance mechanism + organism (catches mechanistic papers)
    '(("resistance mechanism" OR "mechanism of resistance" OR "resistance determinant")[Title/Abstract] AND (bacteria[All Fields] OR organism[All Fields] OR "species"[All Fields]))',
    
    # Generic mutation + bacterial organism patterns
    '((mutation[Title/Abstract] OR "amino acid substitution"[Title/Abstract]) AND (Salmonella[Title/Abstract] OR "Escherichia coli"[Title/Abstract] OR Staphylococcus[Title/Abstract] OR Pseudomonas[Title/Abstract] OR Acinetobacter[Title/Abstract] OR Klebsiella[Title/Abstract] OR Streptococcus[Title/Abstract]) AND (resistance[Title/Abstract] OR susceptibility[Title/Abstract]))',
    
    # Generic resistance + susceptibility determination (MIC/MICROBIOLOGY pattern)
    '(("minimum inhibitory concentration" OR "MIC" OR "antimicrobial susceptibility")[Title/Abstract] AND (mutation[Title/Abstract] OR gene[Title/Abstract]) AND (resistance[Title/Abstract] OR "determination region"[Title/Abstract]))',
    
    # Generic enzyme/protein + resistance mechanism
    '(("beta-lactamase" OR "aminoglycoside modifying" OR "efflux" OR "acetyltransferase" OR "phosphotransferase")[Title/Abstract] AND (mutation[Title/Abstract] OR variant[Title/Abstract] OR resistance[Title/Abstract]))',
    
    # Generic gene expression/regulation + resistance
    '(("gene expression" OR "upregulation" OR "downregulation" OR "overexpression" OR "regulatory")[Title/Abstract] AND ("resistance" OR "antibiotic")[Title/Abstract] AND (bacteria[All Fields] OR organism[All Fields]))',

    # Additional coverage for common resistant pathogens
    '(Pseudomonas aeruginosa[mh] AND ("Drug Resistance, Microbial"[mh] OR "beta-Lactamases"[mh]))',
    '(Acinetobacter baumannii[mh] AND ("Drug Resistance, Microbial"[mh] OR "beta-Lactamases"[mh]))',
    
    # Macrolide/Linezolid resistance
    '(("Macrolides"[mh] OR "Linezolid"[mh]) AND ("Drug Resistance, Microbial"[mh] OR Mutation[mh]))',
    
    # Glycopeptide resistance (Vancomycin)
    '(("Glycopeptides"[mh] OR "Vancomycin"[mh]) AND ("Drug Resistance, Microbial"[mh] OR genetics[mh]))',
    
    # Horizontal gene transfer mechanisms
    '(("Genetic Transfer"[mh] OR "Conjugation, Microbial"[mh] OR "Plasmids"[mh]) AND "Drug Resistance, Microbial"[mh])',
    
    # Transposons and insertion sequences
    '(("Transposons"[mh] OR "DNA, Bacterial"[mh]) AND ("Drug Resistance, Microbial"[mh] OR resistance[All Fields]))',
    
    # ESBL and AmpC producers
    '(("Extended Spectrum beta-Lactamases"[All Fields] OR "AmpC"[All Fields] OR "ESBL"[All Fields]) AND (bacteria[mh] OR genetics[mh]))',
    
    # Anti-Infective Agents as Substance
    '("Anti-Infective Agents"[Substance] AND ("Drug Resistance, Microbial"[mh] OR bacteria[mh]))',
    '("Anti-Infective Agents"[Substance] AND (resistance[All Fields] OR "genetic"[All Fields]))',
    '("Anti-Infective Agents"[Substance] AND (Neisseria meningitidis[mh] OR Streptococcus pneumoniae[mh] OR Haemophilus influenzae[mh] OR Mycobacterium tuberculosis[mh] OR Staphylococcus aureus[mh]))',
    '("Anti-Infective Agents"[Substance] AND ("drug effects"[sh]))',
    '("Anti-Infective Agents"[Substance] AND (susceptibility[All Fields] OR "MIC"[All Fields]))',
    
    # Anti-Bacterial Agents as Substance
    '("Anti-Bacterial Agents"[Substance] AND ("Drug Resistance, Microbial"[mh] OR bacteria[mh]))',
    '("Anti-Bacterial Agents"[Substance] AND (resistance[All Fields] OR "genetic"[All Fields] OR mutation[All Fields]))',
    '("Anti-Bacterial Agents"[Substance] AND (susceptibility[All Fields] OR "MIC"[All Fields] OR "minimum inhibitory concentration"[All Fields]))',
    '("Anti-Bacterial Agents"[Substance] AND ("drug effects"[sh] OR pharmacology[sh]))',
    
    # Anti-Bacterial Agents as MeSH Term (combined with resistance/genetics)
    '("Anti-Bacterial Agents"[mh] AND ("Drug Resistance, Microbial"[mh] OR resistance[All Fields]))',
    '("Anti-Bacterial Agents"[mh] AND (bacteria[mh] OR bacterial[mh] OR genetics[mh]))',
    '("Anti-Bacterial Agents"[mh] AND (Neisseria meningitidis[mh] OR Streptococcus pneumoniae[mh] OR Haemophilus influenzae[mh] OR Pseudomonas aeruginosa[mh] OR Acinetobacter baumannii[mh]))',
    '("Anti-Bacterial Agents"[mh] AND ("drug effects"[sh] OR pharmacology[sh] OR therapeutic use[sh]))',
    '("Anti-Bacterial Agents"[mh] AND (Mutation[mh] OR genetics[mh] OR gene[All Fields]))',
    
    # Tetracyclines
    '("Tetracyclines"[mh] OR "Tetracycline Resistance"[mh]) AND ("Drug Resistance, Microbial"[mh] OR bacteria[mh])',
    '(Tetracycline[mh] AND ("Drug Resistance, Microbial"[mh] OR resistance[All Fields]))',
    '(("Doxycycline"[mh] OR "Minocycline"[mh]) AND ("Drug Resistance, Microbial"[mh] OR genetic[All Fields]))',
    
    # Macrolides (in addition to existing)
    '("Macrolides"[mh] AND ("Drug Resistance, Microbial"[mh] OR bacteria[mh] OR genetics[mh]))',
    '(("Erythromycin"[mh] OR "Azithromycin"[mh] OR "Clarithromycin"[mh]) AND ("Drug Resistance, Microbial"[mh] OR resistance[All Fields]))',
    
    # Sulfonamides
    '("Sulfonamides"[mh] OR "Sulfonamide Resistance"[mh]) AND ("Drug Resistance, Microbial"[mh] OR bacteria[mh])',
    '(Sulfamethoxazole[mh] AND ("Drug Resistance, Microbial"[mh] OR resistance[All Fields]))',
    
    # Trimethoprim and Co-trimoxazole
    '("Trimethoprim"[mh] OR "Trimethoprim-Sulfamethoxazole"[mh]) AND ("Drug Resistance, Microbial"[mh] OR bacteria[mh])',
    '(("Trimethoprim"[mh] OR "TMP-SMX"[All Fields]) AND (resistance[All Fields] OR genetic[All Fields]))',
    
    # Chloramphenicol
    '("Chloramphenicol"[mh] OR "Chloramphenicol Resistance"[mh]) AND ("Drug Resistance, Microbial"[mh] OR bacteria[mh])',
    '(Chloramphenicol[mh] AND (resistance[All Fields] OR genetic[All Fields]))',
    
    # Penicillins (broader coverage)
    '("Penicillins"[mh] AND ("Drug Resistance, Microbial"[mh] OR "beta-Lactamases"[mh]))',
    '(("Amoxicillin"[mh] OR "Ampicillin"[mh] OR "Penicillin G"[mh]) AND ("Drug Resistance, Microbial"[mh] OR resistance[All Fields]))',
    
    # Oxazolidinones
    '(("Oxazolidinones"[mh] OR "Linezolid"[mh]) AND ("Drug Resistance, Microbial"[mh] OR bacteria[mh]))',
    '(Linezolid[mh] AND (resistance[All Fields] OR genetic[All Fields]))',
    
    # Lipopeptides
    '("Lipopeptides"[mh] OR "Daptomycin"[mh]) AND ("Drug Resistance, Microbial"[mh] OR bacteria[mh])',
    
    # Rifamycins
    '("Rifamycins"[mh] OR "Rifampicin"[mh]) AND ("Drug Resistance, Microbial"[mh] OR bacteria[mh])',
    '(Rifampicin[mh] AND (resistance[All Fields] OR genetic[All Fields] OR mutation[All Fields]))',
    
    # Fosfomycin
    '("Fosfomycin"[mh]) AND ("Drug Resistance, Microbial"[mh] OR bacteria[mh] OR resistance[All Fields])',
    
    # Polymyxins
    '(("Polymyxins"[mh] OR "Colistin"[mh] OR "Polymyxin B"[mh]) AND ("Drug Resistance, Microbial"[mh] OR bacteria[mh]))',
    
    # Generic regulatory system searches (catches two-component systems, efflux regulation, etc.)
    '(("two-component" OR "regulatory system" OR "sensory histidine kinase" OR "response regulator" OR "quorum sensing")[Title/Abstract] AND (resistance[Title/Abstract] OR "drug effect"[Title/Abstract]))',
    
    # Generic signal transduction + resistance
    '(("signal transduction" OR "cell envelope" OR "outer membrane" OR "lipopolysaccharide" OR "peptidoglycan")[Title/Abstract] AND (mutation[Title/Abstract] OR resistance[Title/Abstract]))',
    
    # Generic inactivation/modification mechanisms
    '(("enzymatic inactivation" OR "chemical modification" OR "antibiotic inactivation")[Title/Abstract] AND (resistance[Title/Abstract] OR gene[Title/Abstract]))',
    
    # Nitroimidazoles
    '(("Nitroimidazoles"[mh] OR "Metronidazole"[mh]) AND ("Drug Resistance, Microbial"[mh] OR bacteria[mh]))',

]

session = requests.Session()
all_unique_pmids = set()
last_request_time = [time.time()]

start_time = time.time()

for i, base_query in enumerate(search_queries, 1):
    print(f"\n{'='*60}")
    print(f"Query {i}/{len(search_queries)}: {base_query[:80]}...")
    
    query_pmids = set()
    
    for year in range(year_range[0], year_range[1] + 1):
        fetch_pmids_recursive(session, base_query, year, all_pmids=query_pmids)
    
    print(f"Query {i} total: {len(query_pmids)} PMIDs")
    all_unique_pmids.update(query_pmids)
    elapsed = time.time() - start_time
    print(f"Cumulative: {len(all_unique_pmids)} | Elapsed: {elapsed/60:.1f} min")

# --- Save results ---
print(f"\n{'='*60}")
print("SAVING RESULTS")
print(f"{'='*60}")

final_pmid_list = sorted(all_unique_pmids)

if final_pmid_list:
    output_txt_file = "amr_pmids_genetics2.txt"
    with open(output_txt_file, "w") as f:
        f.write("\n".join(final_pmid_list))
    print(f"Success! {len(final_pmid_list)} unique PMIDs saved to {output_txt_file}")

    output_csv_file = "amr_pmids_genetics2.csv"
    df = pd.DataFrame(final_pmid_list, columns=["PMID"])
    df.to_csv(output_csv_file, index=False)
    print(f"Success! {len(final_pmid_list)} unique PMIDs saved to {output_csv_file}")
else:
    print("Failed to retrieve any PMIDs.")

total_time = time.time() - start_time
print(f"\nTotal time: {total_time/60:.1f} minutes | {len(final_pmid_list)} PMIDs collected")



Query 1/87: ("drug resistance, microbial"[MeSH Terms] OR "antimicrobial resistance"[All Fiel...
  1961: 2 PMIDs (total: 2)
  1962: 3 PMIDs (total: 5)
  1963: 20 PMIDs (total: 25)
  1964: 61 PMIDs (total: 86)
  1965: 50 PMIDs (total: 136)
  1966: 65 PMIDs (total: 201)
  1967: 87 PMIDs (total: 288)
  1968: 113 PMIDs (total: 401)
  1969: 129 PMIDs (total: 530)
  1970: 144 PMIDs (total: 674)
  1971: 229 PMIDs (total: 903)
  1972: 271 PMIDs (total: 1174)
  1973: 276 PMIDs (total: 1450)
  1974: 365 PMIDs (total: 1815)
  1975: 295 PMIDs (total: 2110)
  1976: 254 PMIDs (total: 2364)
  1977: 237 PMIDs (total: 2601)
  1978: 209 PMIDs (total: 2810)
  1979: 239 PMIDs (total: 3049)
  1980: 226 PMIDs (total: 3275)
  1981: 176 PMIDs (total: 3451)
  1982: 202 PMIDs (total: 3653)
  1983: 237 PMIDs (total: 3890)
  1984: 249 PMIDs (total: 4139)
  1985: 274 PMIDs (total: 4413)
  1986: 301 PMIDs (total: 4714)
  1987: 275 PMIDs (total: 4989)
  1988: 337 PMIDs (total: 5326)
  1989: 371 PMIDs (total: 5697)
 

In [12]:
file1 = "./amrprofiler_database/amr_genes_pmids_amrprofiler_uniq.txt"
file2 = "amr_pmids_genetics2.txt"

with open(file1) as f1:
    pmids1 = set(line.strip() for line in f1 if line.strip())

with open(file2) as f2:
    pmids2 = set(line.strip() for line in f2 if line.strip())

missing_pmids = pmids1 - pmids2

print(f"PMIDs in {file1} but not in {file2}: {len(missing_pmids)}")
for pmid in sorted(missing_pmids):
    print(pmid)

PMIDs in ./amrprofiler_database/amr_genes_pmids_amrprofiler_uniq.txt but not in amr_pmids_genetics2.txt: 196
1048253
1063935
10722565
108177
1081773
1099185
1157715
1173537
1185024
1212191
1238435
1238438
1265429
1265473
1270568
127609
1376392
1451064
15272191
1556186
1572895
1585548
1585554
16042375
1612706
16333139
165843
16678922
1676919
1678572
1680145
16948925
17210666
1726153
1728454
1730292
1737687
17379729
17536925
1804759
1839104
1848034
1850092
1916415
1917836
19535342
19560485
1973801
200386180
20129969
2048245
20682255
2080539
2083358
2135985
2149706
2166591
2210622
22146011
2215583
2233091
2244224
225263
2290816
22999884
2301472
2311476
2345949
23469330
2358795
23723398
23734212
23796922
2399702
2418587
24336371
24426128
2459875
24625868
24684968
2468497
2484587
25635013
25643999
25814612
25931589
25953190
2602991
2603372
26227599
2624836
2628334
2637787
2639249
2640906
2653124
2667426
2674553
26999202
2716164
2716831
2719171
275724
27747296
2779539
2779921
2791598
2810676

In [13]:
file1 = "/home/argis/Desktop/austin/pubmed_amr/amrprofiler_database/amr_mutations_pmids_amrprofiler_uniq.txt"
file2 = "amr_pmids_genetics2.txt"
    
with open(file1) as f1:
    pmids1 = set(line.strip() for line in f1 if line.strip())

with open(file2) as f2:
    pmids2 = set(line.strip() for line in f2 if line.strip())

missing_pmids = pmids1 - pmids2

print(f"PMIDs in {file1} but not in {file2}: {len(missing_pmids)}")
for pmid in sorted(missing_pmids):
    print(pmid)

PMIDs in /home/argis/Desktop/austin/pubmed_amr/amrprofiler_database/amr_mutations_pmids_amrprofiler_uniq.txt but not in amr_pmids_genetics2.txt: 5
1850092
193837278
2550546
29250039
35184552
